# Testing Model Usage with Groq API

## I. Install necessary packages (if not already installed)

In [20]:
# %pip install groq
# %pip install google-adk
# %pip install litellm
# %pip install "pymupdf4llm[ocr,layout]"

### Sanity Check for Packages
Main, compatible library versions (as of Feb 4, 2026):
- google-adk==1.23.0
- groq==1.0.0
- litellm==1.81.7

In [1]:
%pip list

Package                                  Version
---------------------------------------- -----------
absl-py                                  2.3.1
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
alembic                                  1.18.3
annotated-doc                            0.0.4
annotated-types                          0.7.0
anyio                                    4.12.1
appnope                                  0.1.4
asttokens                                3.0.0
attrs                                    25.4.0
Authlib                                  1.6.6
certifi                                  2025.10.5
cffi                                     2.0.0
charset-normalizer                       3.4.4
click                                    8.3.1
cloudpickle                              3.1.2
coloredlogs                              15

In [21]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


## II. Import and Setup

In [5]:
import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any

from dotenv import load_dotenv
from groq import Groq

from google.genai import types
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner

import pymupdf4llm
import pymupdf.layout


### Setup 

In [6]:
load_dotenv()
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

if client:
    print("API Key Loaded!")

API Key Loaded!


Configure retry options

In [7]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

### Test API

Simple model calling via Groq API

In [6]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in natural language processing (NLP) as they enable efficient and effective processing of large amounts of text data. The importance of fast language models can be attributed to several factors:

1. **Speed and Efficiency**: Fast language models can process text data quickly, making them ideal for real-time applications, such as language translation, chatbots, and sentiment analysis. This speed enables systems to respond promptly to user inputs, enhancing the overall user experience.
2. **Scalability**: Fast language models can handle large volumes of text data, making them suitable for big data applications, such as text classification, named entity recognition, and topic modeling. This scalability enables organizations to analyze large datasets and gain valuable insights.
3. **Real-time Decision Making**: Fast language models can facilitate real-time decision making in applications, such as sentiment analysis, opinion mining, and recommendation system

Creating an agent via Groq, ADK, and LiteLLM

In [22]:
model = LiteLlm(
    model="groq/llama-3.3-70b-versatile",
    num_retries=5
)

agent_llama = Agent(
    name="greeting_agent",
    model=model,
    description="A helpful assistant that greets users.",
    instruction="""
        You are a helpful assistant that greets users. Ask for the user's name and greet them by name.
    """,
)

In [23]:
runner = InMemoryRunner(agent=agent_llama)

In [24]:
response = await runner.run_debug("Hello there.", verbose=True)


 ### Created new session: debug_session_id

User > Hello there.
greeting_agent > Hello. I'm the greeting_agent, a helpful assistant that greets users. It's nice to meet you. Before we get started, could you please tell me your name? I'd love to greet you personally.


## III. Load and Prepare CUAD Dataset

Load SQuAD-formatted JSON data first and create pandas DataFrame

Helper function

In [31]:
# Parse the JSON and create a DataFrame
def parse_cuad_json(cuad_data: Dict[str, Any]) -> pd.DataFrame:
    """
    Parse CUAD JSON data into a pandas DataFrame.
    
    Each row represents a single question-answer pair with columns:
    - contract_title: Title of the contract
    - context: The paragraph/context text
    - question: The question about the clause
    - ground_truth_answer: The answer text (empty string if no answer)
    """
    rows = []
    
    for document in cuad_data['data']:
        contract_title = document['title']
        
        for paragraph in document['paragraphs']:
            context = paragraph['context']
            
            for qa in paragraph['qas']:
                question = qa['question']
                is_impossible = qa.get('is_impossible', False)
                
                # Handle cases with no answer
                if is_impossible or len(qa['answers']) == 0:
                    ground_truth_answer = ""
                else:
                    # Use the first answer's text
                    ground_truth_answer = qa['answers'][0]['text']
                
                rows.append({
                    'contract_title': contract_title,
                    'context': context,
                    'question': question,
                    'ground_truth_answer': ground_truth_answer
                })
    
    return pd.DataFrame(rows)

In [32]:
# Load CUAD_v1.json file
cuad_json_path = "../datasets/CUAD_v1/CUAD_v1.json"

print("Loading CUAD dataset...")
with open(cuad_json_path, 'r', encoding='utf-8') as f:
    cuad_data = json.load(f)

print(f"✓ CUAD dataset loaded successfully")
print(f"  Number of documents: {len(cuad_data['data'])}")


Loading CUAD dataset...
✓ CUAD dataset loaded successfully
  Number of documents: 510


Read JSON file

In [33]:
# Create the DataFrame
print("\nParsing CUAD data into DataFrame...")
df_cuad = parse_cuad_json(cuad_data)

print(f"✓ DataFrame created successfully")
print(f"  Total question-answer pairs: {len(df_cuad)}")
print(f"  Questions with no answer: {(df_cuad['ground_truth_answer'] == '').sum()}")
print(f"  Questions with answer: {(df_cuad['ground_truth_answer'] != '').sum()}")


Parsing CUAD data into DataFrame...
✓ DataFrame created successfully
  Total question-answer pairs: 20910
  Questions with no answer: 14208
  Questions with answer: 6702


In [34]:
print("\nFirst 5 rows of the dataset:")
df_cuad.head()


First 5 rows of the dataset:


,contract_title,context,question,ground_truth_answer
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,DISTRIBUTOR AGREEMENT
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,Distributor
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,"7th day of September, 1999."
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The term of this Agreement shall be ten (10)...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The term of this Agreement shall be ten (10)...


### Grab small sample of dataset for quick agent testing.

In [ ]:
sample_size = 10
df_sample = df_cuad[df_cuad['ground_truth_answer'] != ''].head(sample_size).copy()
print(f"Example question: {df_sample['context'][3]}")
df_sample

Example question: EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  wil

,contract_title,context,question,ground_truth_answer
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,DISTRIBUTOR AGREEMENT
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,Distributor
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,"7th day of September, 1999."
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The term of this Agreement shall be ten (10)...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The term of this Agreement shall be ten (10)...
5,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,If Distributor comp...
7,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,This Agreement is to be construed according to...
10,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,The Distributor shall not order or ...
11,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,Distributor further agrees that it will not in...
13,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,Highlight the parts (if any) of this contract ...,During the Term of this Agreement and for a pe...


Aggregate questions by contract title and context.


## IV. Task basic model with answering questions.

A basic agent that receives a contract and answers questions based solely on its content.

- TODO: Use ADK's Artifacts?

In [68]:
def create_contract_qa_agent(contract_title: str, contract_text: str):
    """
    Create an agent specialized for answering questions about a specific contract.
    
    Args:
        contract_title: The title/name of the contract
        contract_text: The full text content of the contract
    
    Returns:
        An Agent configured to answer questions about this contract
    """
    
    system_instruction = f"""You are a legal contract analyst. Your task is to answer questions about the following contract.

CONTRACT TITLE: {contract_title}

CONTRACT TEXT:
{contract_text}

IMPORTANT RULES:
1. Answer questions ONLY based on the information explicitly stated in the contract above.
2. If the answer cannot be found in the contract, respond with "Not found in contract."
3. Quote relevant text from the contract when possible to support your answer.
4. Be precise and concise in your answers.
5. Do not make assumptions or infer information that is not explicitly stated.
"""
    
    model = LiteLlm(
        model="groq/llama-3.3-70b-versatile",
        num_retries=5
    )
    
    agent = Agent(
        name="contract_qa_agent",
        model=model,
        description="An agent that answers questions about a specific contract based solely on its content.",
        instruction=system_instruction,
    )
    
    return agent

In [69]:
async def answer_contract_questions(
    contract_title: str, 
    contract_text: str, 
    questions: List[str],
    verbose: bool = False
) -> Dict[str, str]:
    """
    Answer a list of questions about a contract.
    
    Args:
        contract_title: The title of the contract
        contract_text: The full text of the contract
        questions: List of questions to answer
        verbose: If True, print debug information
    
    Returns:
        Dictionary mapping each question to its answer
    """
    # Create the agent with the contract embedded in its instructions
    agent = create_contract_qa_agent(contract_title, contract_text)
    
    # Create a runner for this agent
    runner = InMemoryRunner(agent=agent)
    
    answers = {}
    
    for i, question in enumerate(questions, 1):
        if verbose:
            print(f"\n{'='*60}")
            print(f"Question {i}/{len(questions)}: {question[:100]}...")
        
        # Run the agent with this question
        response = await runner.run_debug(question, verbose=verbose)
        
        # Extract the answer from the response
        # The response object contains the agent's reply
        answer = response.content if hasattr(response, 'content') else str(response)
        answers[question] = answer
        
        if verbose:
            print(f"Answer: {answer[:200]}..." if len(answer) > 200 else f"Answer: {answer}")
    
    return answers

### Test the Contract Q&A Agent

Let's test with a sample contract from our dataset.

In [70]:
grouped_df = df_sample.groupby(['contract_title', 'context'])['question'].apply(list).reset_index()

Check the structure and questions for the sample contract

In [74]:
# Get the first contract from our grouped sample data
sample_contract = grouped_df.iloc[0]

test_contract_title = sample_contract['contract_title']
test_contract_text = sample_contract['context']
test_questions = sample_contract['question']

print(f"Contract Title: {test_contract_title}")
print(f"Contract Text Length: {len(test_contract_text)} characters")
print(f"Number of Questions: {len(test_questions)}")
print(f"\nFirst 3 Questions:")
for i, q in enumerate(test_questions[:3], 1):
    print(f"  {i}. {q[:80]}...")

Contract Title: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Contract Text Length: 54290 characters
Number of Questions: 10

First 3 Questions:
  1. Highlight the parts (if any) of this contract related to "Document Name" that sh...
  2. Highlight the parts (if any) of this contract related to "Parties" that should b...
  3. Highlight the parts (if any) of this contract related to "Agreement Date" that s...


In [75]:
# Test with just the first 3 questions to start simple
test_subset = test_questions[:3]

print("Running Contract Q&A Agent...")
print("=" * 60)

results = await answer_contract_questions(
    contract_title=test_contract_title,
    contract_text=test_contract_text,
    questions=test_subset,
    verbose=True
)

Running Contract Q&A Agent...

Question 1/3: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by ...

 ### Created new session: debug_session_id

User > Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
contract_qa_agent > The contract is referred to as the "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT" and also as the "DISTRIBUTOR AGREEMENT" within the document. 

The parts of the contract related to the "Document Name" that should be reviewed by a lawyer are:

1. The title of the contract: "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT" - This should be reviewed to ensure it accurately reflects the contents and purpose of the agreement.
2. Section 6.5: "Entire Agreement" - This section states that the agreement supersedes all other agreements between the parties, which could have implications for the document name and any related c

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kgezc6x5egfa7gfdeh9889tq` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10371, Requested 10828. Please try again in 45.995s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
 LiteLLM Retried: 5 times

In [ ]:
# Display results in a clean format
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

for i, (question, answer) in enumerate(results.items(), 1):
    print(f"\n--- Question {i} ---")
    print(f"Q: {question}")
    print(f"\nA: {answer}")
    print("-" * 40)

### Alternative: Simple Version (Direct API Call)

If you want something even simpler without the ADK framework, here's a direct Groq API version:

In [ ]:
def answer_questions_simple(
    contract_title: str,
    contract_text: str,
    questions: List[str],
    client: Groq = None
) -> Dict[str, str]:
    """
    Simple version using direct Groq API calls.
    Answers all questions in a single API call for efficiency.
    
    Args:
        contract_title: The title of the contract
        contract_text: The full text of the contract  
        questions: List of questions to answer
        client: Groq client (uses global client if not provided)
    
    Returns:
        Dictionary mapping each question to its answer
    """
    if client is None:
        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    
    # Format questions as a numbered list
    questions_text = "\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)])
    
    system_prompt = """You are a legal contract analyst. Your task is to answer questions about contracts.

IMPORTANT RULES:
1. Answer questions ONLY based on the information explicitly stated in the contract provided.
2. If the answer cannot be found in the contract, respond with "Not found in contract."
3. Quote relevant text from the contract when possible to support your answer.
4. Be precise and concise in your answers.
5. Do not make assumptions or infer information that is not explicitly stated.
6. Format your response as a numbered list matching the question numbers."""

    user_prompt = f"""CONTRACT TITLE: {contract_title}

CONTRACT TEXT:
{contract_text}

QUESTIONS TO ANSWER:
{questions_text}

Please answer each question based solely on the contract text above."""

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.1,  # Low temperature for more consistent, factual responses
    )
    
    return response.choices[0].message.content

In [ ]:
# Test the simple version
print("Testing Simple Direct API Version...")
print("=" * 60)

simple_results = answer_questions_simple(
    contract_title=test_contract_title,
    contract_text=test_contract_text,
    questions=test_subset  # Using same 3 questions as before
)

print("\nAgent Response:")
print("-" * 40)
print(simple_results)

## V. Rate-Limit-Aware Contract Q&A

The previous version hit Groq's free tier limit (12k TPM). Here's an improved version with:
1. Token estimation before sending
2. Automatic retry with delays on rate limit
3. Option to truncate long contracts for testing

In [76]:
import time

def estimate_tokens(text: str) -> int:
    """Rough token estimate (~4 chars per token for English)."""
    return len(text) // 4

def truncate_contract(contract_text: str, max_tokens: int = 6000) -> str:
    """Truncate contract to fit within token limit."""
    max_chars = max_tokens * 4  # Rough estimate
    if len(contract_text) <= max_chars:
        return contract_text
    return contract_text[:max_chars] + "\n\n[... CONTRACT TRUNCATED FOR TESTING ...]"

def answer_contract_questions_v2(
    contract_title: str,
    contract_text: str,
    questions: List[str],
    client: Groq = None,
    model: str = "llama-3.3-70b-versatile",
    max_retries: int = 3,
    retry_delay: int = 60,
    truncate_for_testing: bool = False
) -> Dict[str, Any]:
    """
    Answer questions about a contract with rate limit handling.
    
    Args:
        contract_title: The title of the contract
        contract_text: The full text of the contract  
        questions: List of questions to answer
        client: Groq client
        model: Model to use
        max_retries: Retries on rate limit
        retry_delay: Seconds to wait before retry
        truncate_for_testing: If True, truncate long contracts to avoid rate limits
    
    Returns:
        Dictionary with 'answers', 'usage', and 'truncated' flag
    """
    if client is None:
        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    
    # Optionally truncate for testing
    truncated = False
    if truncate_for_testing:
        original_len = len(contract_text)
        contract_text = truncate_contract(contract_text, max_tokens=6000)
        truncated = len(contract_text) < original_len
        if truncated:
            print(f"📝 Contract truncated from {original_len} to {len(contract_text)} chars for testing")
    
    # Format questions
    questions_text = "\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)])
    
    system_prompt = """You are a legal contract analyst. Answer questions about the contract.

RULES:
1. Answer ONLY based on information explicitly in the contract.
2. If not found, say "Not found in contract."
3. Quote relevant text when possible.
4. Be concise. Format as numbered list."""

    user_prompt = f"""CONTRACT: {contract_title}

{contract_text}

QUESTIONS:
{questions_text}"""

    # Estimate tokens
    est_input = estimate_tokens(system_prompt + user_prompt)
    print(f"📊 Estimated input tokens: ~{est_input}")
    
    if est_input > 10000:
        print(f"⚠️  Warning: ~{est_input} tokens exceeds safe limit for free tier (12k TPM)")
        print(f"   Consider using truncate_for_testing=True or waiting between requests")
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                model=model,
                temperature=0.1,
            )
            
            usage = response.usage
            print(f"✅ Success! Actual tokens - Input: {usage.prompt_tokens}, Output: {usage.completion_tokens}")
            
            return {
                "answers": response.choices[0].message.content,
                "usage": {
                    "prompt_tokens": usage.prompt_tokens,
                    "completion_tokens": usage.completion_tokens,
                    "total_tokens": usage.total_tokens
                },
                "truncated": truncated
            }
            
        except Exception as e:
            error_str = str(e).lower()
            if "rate_limit" in error_str or "rate limit" in error_str:
                # Try to extract wait time from error message
                wait_time = retry_delay * (attempt + 1)
                print(f"⏳ Rate limit hit. Waiting {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                raise e
    
    raise Exception(f"Failed after {max_retries} retries due to rate limiting")

### Test with truncation (safe for free tier)

In [77]:
# Test with truncation enabled - safe for free tier
# This will cut the contract to ~6000 tokens to stay under the 12k TPM limit

print("Testing with TRUNCATED contract (safe for free tier)...")
print("=" * 60)

result = answer_contract_questions_v2(
    contract_title=test_contract_title,
    contract_text=test_contract_text,
    questions=test_subset,  # First 3 questions
    truncate_for_testing=True  # Truncate to fit rate limit
)

print("\n" + "=" * 60)
print("ANSWERS:")
print("=" * 60)
print(result["answers"])

if result["truncated"]:
    print("\n⚠️  Note: Contract was truncated. Some answers may be incomplete.")

Testing with TRUNCATED contract (safe for free tier)...
📝 Contract truncated from 54290 to 24042 chars for testing
📊 Estimated input tokens: ~6210
✅ Success! Actual tokens - Input: 4779, Output: 249

ANSWERS:
Here are the answers to your questions:

1. The part of this contract related to "Document Name" is:
   * "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT" 
   as stated in "CONTRACT: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT".

2. The parts of this contract related to "Parties" are:
   * "Electric City Corp., a Delaware corporation ('Company')" 
   * "Electric City of Illinois LLC ('Distributor')" 
   as stated in "THIS DISTRIBUTOR AGREEMENT (the 'Agreement') is made by and between Electric City Corp., a Delaware corporation ('Company') and Electric City of Illinois LLC ('Distributor')".

3. The part of this contract related to "Agreement Date" is:
   * "this 7th day of September, 1999" 
   as stated in "THIS DISTRIBUTOR AGREEMENT (the 'Agreement') is made by and be

### Test with full contract (may hit rate limit)

If you want to test with the full contract, either:
1. Wait 60+ seconds between runs
2. Upgrade to Groq Dev Tier ($5/month for higher limits)
3. Use a different provider (OpenAI, Anthropic, etc.)

In [ ]:
# Test with FULL contract (will retry automatically on rate limit)
# This will wait and retry if rate limited

print("Testing with FULL contract...")
print("=" * 60)
print("⚠️  If rate limited, will wait up to 3 minutes total")
print()

result_full = answer_contract_questions_v2(
    contract_title=test_contract_title,
    contract_text=test_contract_text,
    questions=test_subset,
    truncate_for_testing=False,  # Use full contract
    retry_delay=60  # Wait 60s between retries
)

print("\n" + "=" * 60)
print("ANSWERS (Full Contract):")
print("=" * 60)
print(result_full["answers"])

# VI. Testing Agent Using Memory
This part aims to resolve the problem of overloading context window when trying to load in the contract to answer questions.

Planned Approach: HybridRAG (Vectors + KnowledgeGraph)

The Architecture: "The Router Pattern"You don't force one database to do everything. Instead, you build a "Router" (a simple classifier) that sits between the User and your Data.
- Brain 1 (Vector DB): Good at unstructured questions ("What are the penalties?", "Summarize the confidentiality clause").
- Brain 2 (Knowledge Graph): Good at structured questions ("Draw the relationship between the Client and the Vendor", "Visualize the termination timeline").

The Workflow
1. User Query: "Show me a diagram of the payment milestones."
2. Router (Small LLM): Analyzes the intent.Is this a text lookup? $\rightarrow$ Route to Vector DB.Is this a visualization request? $\rightarrow$ Route to Knowledge Graph.
3. Agent: Receives the data from the chosen source and generates the final answer (or Mermaid code).

## Read contract PDFs as input

Start by reading PDF as input using pymupdf4llm

In [8]:
# Convert to markdown
md_text = pymupdf4llm.to_markdown("../datasets/CUAD_v1/full_contract_pdf/Part_III/Distributor/LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT.PDF")

import pathlib

pathlib.Path("../generated_files/contracts_md/sample_contract_limeenergy.md").write_bytes(md_text.encode())

39015